# Lab 06 — Credit Risk + Fairness Basics
**Responsible ML Track** · Intermediate · ~50 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Compute group-wise accuracy and TPR
2. Calculate disparate impact ratio
3. Understand why equal accuracy ≠ fair
4. Sketch a reweighing mitigation

## Datasets (this folder)
- `german_credit_prepared.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-02-classical-machine-learning/labs/lab-06-credit-fairness-preview/lab-06-credit-fairness-preview.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0** first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 2).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`german_credit_prepared.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-06-credit-fairness-preview"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-02-classical-machine-learning/labs/lab-06-credit-fairness-preview/bundle/dataset.zip"
NEED = ["german_credit_prepared.csv"]  # unzipped files used by the code below

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Intermediate: When the Scorecard Plays Favorites (Fairness Preview)

> **Scenario:** The bank's legal team reviews your credit model and asks a uncomfortable question: "Does it treat male and female applicants fairly?" Overall accuracy looks fine — 74.5%. But averages can hide a lot. You dig into performance *by group* and find the model approves female applicants at a lower rate than males, even though their default rates are similar. This lab shows why "equal accuracy" is not the same as "fair", and previews how to measure and mitigate the gap.
>
> **You will learn:** how to compute group-wise accuracy and true-positive rate (TPR), what the disparate impact ratio measures and why 0.8 is a common legal benchmark, and one concrete mitigation — per-group decision thresholds.
> **Time:** ~50 minutes. **Level:** Intermediate. **Needs:** pandas + sklearn. **Env:** 🟢 Colab only.

### Mental map

| Concept | One-line idea | Where it shows up |
|---|---|---|
| Group-wise metrics | Accuracy/TPR computed separately per demographic group | female vs male TPR |
| Disparate impact ratio | Ratio of approval rates between groups | DI < 0.8 raises legal flags |
| Equal accuracy ≠ fair | Two groups can have similar accuracy but very different error types | TPR/FPR gaps |
| Per-group thresholds | Different score cutoffs for different groups | A simple mitigation lever |

---

## 1. Group-wise metrics: accuracy and TPR by sex

A single overall accuracy can hide very different experiences for different groups. The first fairness step is always to **slice the metrics by group**. Here we train the same kind of credit model as Lab 05 and compute accuracy and the true-positive rate (TPR — fraction of actual defaulters the model flags) separately for female and male applicants.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix

df = pd.read_csv(r"D:\downloads-D\vilabs-images\ailab-vilabs\course-02-classical-machine-learning\datasets\german_credit_prepared.csv")
y = (df["default"] == "Default").astype(int)
X = df.drop(columns=["default"])
cat_cols = X.select_dtypes("object").columns.tolist()
num_cols = X.select_dtypes("number").columns.tolist()

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])
clf = make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

res = pd.DataFrame({"sex": X_te["sex"].values, "y": y_te.values, "pred": y_pred})
print(f"overall accuracy: {accuracy_score(y_te, y_pred):.4f}\n")
for sex, g in res.groupby("sex"):
    tn, fp, fn, tp = confusion_matrix(g["y"], g["pred"]).ravel()
    acc = accuracy_score(g["y"], g["pred"])
    tpr = tp / (tp + fn)
    fpr = fp / (fp + tn)
    approval = (g["pred"] == 0).mean()
    print(f"{sex:6s}  n={len(g):3d}  acc={acc:.4f}  TPR={tpr:.4f}  FPR={fpr:.4f}  approval_rate={approval:.4f}")


**What to notice:**
1. Overall accuracy (≈0.745) looks respectable — but it blends two different stories.
2. Female TPR (≈0.85) is actually *higher* than male (≈0.775): the model catches a larger share of female defaulters.
3. Yet the female *approval rate* (≈0.49) is lower than male (≈0.60): fewer women are approved overall. That gap is the fairness problem.

> **Pitfall:** `sex` is used here only to *audit* the model, not as a model input. Feeding protected attributes into the model is both illegal in many jurisdictions and defeats the purpose of measuring their effect.

---

## 2. The disparate impact ratio

The **disparate impact (DI) ratio** compares approval rates between groups: DI = (approval rate of the disadvantaged group) / (approval rate of the advantaged group). A DI of 1.0 means perfect parity. In US employment law, DI < 0.8 is the "four-fifths rule" threshold that raises legal scrutiny — and it is a useful benchmark anywhere.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

df = pd.read_csv(r"D:\downloads-D\vilabs-images\ailab-vilabs\course-02-classical-machine-learning\datasets\german_credit_prepared.csv")
y = (df["default"] == "Default").astype(int)
X = df.drop(columns=["default"])
cat_cols = X.select_dtypes("object").columns.tolist()
num_cols = X.select_dtypes("number").columns.tolist()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])
clf = make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

res = pd.DataFrame({"sex": X_te["sex"].values, "y": y_te.values, "pred": y_pred})
f = res[res["sex"] == "female"]
m = res[res["sex"] == "male"]
di = (f["pred"] == 0).mean() / (m["pred"] == 0).mean()
print(f"female approval rate: {(f['pred'] == 0).mean():.4f}")
print(f"male approval rate:   {(m['pred'] == 0).mean():.4f}")
print(f"disparate impact ratio (female/male): {di:.4f}")
print(f"four-fifths rule (>= 0.8)? {'PASS' if di >= 0.8 else 'FAIL'}")

# base rates for context
print("\nactual default rate by sex (test):")
print(res.groupby("sex")["y"].mean().round(4))


**What to notice:**
1. DI ≈ 0.82 — just under the 0.8 four-fifths threshold, so this model would draw legal scrutiny.
2. The gap is not explained by base rates: female default rate (≈0.33) is only slightly higher than male (≈0.29), yet the approval gap is much larger.
3. DI is a *ratio of rates*, so it is comparable across datasets with different overall approval levels.

> **Pitfall:** DI measures *outcome* parity, not *error* parity. A model can have DI = 1.0 and still miss defaulters in one group much more often. Always look at TPR/FPR gaps alongside DI.

---

## 3. Why equal accuracy ≠ fair, and one mitigation

Here is the trap: overall accuracy is nearly identical across groups (≈0.72 female vs ≈0.76 male), so a naive audit says "fair". But the *errors* are distributed unevenly — women are approved less often. Accuracy is a single number that averages over both classes; fairness lives in the details.

One concrete mitigation is **per-group thresholds**: use a different score cutoff for each group so that approval rates (or TPRs) match. This is simple, transparent, and easy to explain to regulators — though it is not the only option, and each has trade-offs.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

df = pd.read_csv(r"D:\downloads-D\vilabs-images\ailab-vilabs\course-02-classical-machine-learning\datasets\german_credit_prepared.csv")
y = (df["default"] == "Default").astype(int)
X = df.drop(columns=["default"])
cat_cols = X.select_dtypes("object").columns.tolist()
num_cols = X.select_dtypes("number").columns.tolist()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])
clf = make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
clf.fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]
sex_te = X_te["sex"].values

print("per-group threshold sweep (raise female threshold to equalize approval):")
for t_f in [0.5, 0.55, 0.6, 0.65]:
    yp = np.where(sex_te == "female", proba >= t_f, proba >= 0.5).astype(int)
    res = pd.DataFrame({"sex": sex_te, "y": y_te.values, "pred": yp})
    f = res[res["sex"] == "female"]
    m = res[res["sex"] == "male"]
    di = (f["pred"] == 0).mean() / (m["pred"] == 0).mean()
    acc = (res["pred"] == res["y"]).mean()
    print(f"  t_female={t_f}: DI={di:.4f}  accuracy={acc:.4f}  "
          f"appr_f={(f['pred'] == 0).mean():.4f}  appr_m={(m['pred'] == 0).mean():.4f}")


**What to notice:**
1. Raising the female threshold to 0.6 lifts DI from ≈0.82 to ≈0.96 — near parity — while overall accuracy stays ≈0.75.
2. The cost is a lower female TPR (fewer female defaulters caught): mitigation trades one fairness goal against another.
3. There is no free lunch — the "right" threshold depends on whether the business prioritizes approval parity, TPR parity, or accuracy.

> **Pitfall:** Per-group thresholds are a *post-hoc* fix. The deeper question is why the model scores women lower in the first place — often the training data itself reflects historical bias. Fixing the data beats patching the threshold.

---

## Exercises (do these!)

### Exercise 1 — TPR by sex
Train the balanced logistic regression from section 1 and compute TPR separately for female and male applicants on the test set. Which group has the higher TPR?
*Expected: female TPR ≈ 0.8500 (n=61), male TPR ≈ 0.7750 (n=139) — female is higher.*

**Follow-up:** What are the false-positive rates? Check: female FPR ≈ 0.3415, male FPR ≈ 0.2525 — women are also falsely flagged more often.

<details>
<summary>Hint</summary>

`confusion_matrix(g["y"], g["pred"]).ravel()` returns `(tn, fp, fn, tp)`; TPR = tp / (tp + fn).

</details>

### Exercise 2 — Disparate impact ratio
Compute the DI ratio (female approval rate / male approval rate) on the test set. Does it pass the four-fifths rule (DI ≥ 0.8)?
*Expected: DI ≈ 0.8236 — just below 0.8, so it FAILS the four-fifths rule.*

**Follow-up:** What are the actual default rates by sex? Check: female ≈ 0.3279, male ≈ 0.2878 — the approval gap is much larger than the base-rate gap.

<details>
<summary>Hint</summary>

DI = `(f["pred"] == 0).mean() / (m["pred"] == 0).mean()` where `f` and `m` are the per-group test frames.

</details>

### Exercise 3 — One mitigation idea
Using the per-group threshold sweep from section 3, find a female threshold that brings DI close to 1.0 while keeping overall accuracy near its original value. Report the threshold, DI, and accuracy.
*Expected: t_female = 0.6 gives DI ≈ 0.9609 and accuracy ≈ 0.7500 (DI near parity, accuracy unchanged).*

**Follow-up:** What happens to female TPR at t_female = 0.6? Check: it drops from ≈0.85 to ≈0.75 — the mitigation trades TPR parity for approval parity.

<details>
<summary>Hint</summary>

`np.where(sex_te == "female", proba >= t_f, proba >= 0.5)` applies different thresholds per group.

</details>

---

## Solutions

Try for 15 min each before peeking.

In [ ]:
# --- Solution 1 ---
import matplotlib
matplotlib.use("Agg")
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix

df = pd.read_csv(r"D:\downloads-D\vilabs-images\ailab-vilabs\course-02-classical-machine-learning\datasets\german_credit_prepared.csv")
y = (df["default"] == "Default").astype(int)
X = df.drop(columns=["default"])
cat_cols = X.select_dtypes("object").columns.tolist()
num_cols = X.select_dtypes("number").columns.tolist()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])
clf = make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)
res = pd.DataFrame({"sex": X_te["sex"].values, "y": y_te.values, "pred": y_pred})
for sex, g in res.groupby("sex"):
    tn, fp, fn, tp = confusion_matrix(g["y"], g["pred"]).ravel()
    print(f"{sex}: TPR={tp/(tp+fn):.4f}")

# --- Solution 2 ---
import matplotlib
matplotlib.use("Agg")
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

df = pd.read_csv(r"D:\downloads-D\vilabs-images\ailab-vilabs\course-02-classical-machine-learning\datasets\german_credit_prepared.csv")
y = (df["default"] == "Default").astype(int)
X = df.drop(columns=["default"])
cat_cols = X.select_dtypes("object").columns.tolist()
num_cols = X.select_dtypes("number").columns.tolist()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])
clf = make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)
res = pd.DataFrame({"sex": X_te["sex"].values, "y": y_te.values, "pred": y_pred})
f = res[res["sex"] == "female"]
m = res[res["sex"] == "male"]
di = (f["pred"] == 0).mean() / (m["pred"] == 0).mean()
print(f"DI = {di:.4f}")

# --- Solution 3 ---
import matplotlib
matplotlib.use("Agg")
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

df = pd.read_csv(r"D:\downloads-D\vilabs-images\ailab-vilabs\course-02-classical-machine-learning\datasets\german_credit_prepared.csv")
y = (df["default"] == "Default").astype(int)
X = df.drop(columns=["default"])
cat_cols = X.select_dtypes("object").columns.tolist()
num_cols = X.select_dtypes("number").columns.tolist()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])
clf = make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
clf.fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]
sex_te = X_te["sex"].values
yp = np.where(sex_te == "female", proba >= 0.6, proba >= 0.5).astype(int)
res = pd.DataFrame({"sex": sex_te, "y": y_te.values, "pred": yp})
f = res[res["sex"] == "female"]
m = res[res["sex"] == "male"]
di = (f["pred"] == 0).mean() / (m["pred"] == 0).mean()
acc = (res["pred"] == res["y"]).mean()
print(f"t_female=0.6: DI={di:.4f}, accuracy={acc:.4f}")

# --- Follow-up 1 ---
import matplotlib
matplotlib.use("Agg")
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix

df = pd.read_csv(r"D:\downloads-D\vilabs-images\ailab-vilabs\course-02-classical-machine-learning\datasets\german_credit_prepared.csv")
y = (df["default"] == "Default").astype(int)
X = df.drop(columns=["default"])
cat_cols = X.select_dtypes("object").columns.tolist()
num_cols = X.select_dtypes("number").columns.tolist()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])
clf = make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)
res = pd.DataFrame({"sex": X_te["sex"].values, "y": y_te.values, "pred": y_pred})
f = res[res["sex"] == "female"]
m = res[res["sex"] == "male"]
tn, fp, fn, tp = confusion_matrix(f["y"], f["pred"]).ravel()
fpr_f = fp / (fp + tn)
tn, fp, fn, tp = confusion_matrix(m["y"], m["pred"]).ravel()
fpr_m = fp / (fp + tn)
assert abs(fpr_f - 0.3415) < 0.01 and abs(fpr_m - 0.2525) < 0.01
print(f"FPR female={fpr_f:.4f}, male={fpr_m:.4f}")

# --- Follow-up 2 ---
import matplotlib
matplotlib.use("Agg")
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

df = pd.read_csv(r"D:\downloads-D\vilabs-images\ailab-vilabs\course-02-classical-machine-learning\datasets\german_credit_prepared.csv")
y = (df["default"] == "Default").astype(int)
X = df.drop(columns=["default"])
cat_cols = X.select_dtypes("object").columns.tolist()
num_cols = X.select_dtypes("number").columns.tolist()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])
clf = make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)
res = pd.DataFrame({"sex": X_te["sex"].values, "y": y_te.values, "pred": y_pred})
rates = res.groupby("sex")["y"].mean()
assert abs(rates["female"] - 0.3279) < 0.01 and abs(rates["male"] - 0.2878) < 0.01
print("default rates:", rates.round(4).to_dict())

# --- Follow-up 3 ---
import matplotlib
matplotlib.use("Agg")
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

df = pd.read_csv(r"D:\downloads-D\vilabs-images\ailab-vilabs\course-02-classical-machine-learning\datasets\german_credit_prepared.csv")
y = (df["default"] == "Default").astype(int)
X = df.drop(columns=["default"])
cat_cols = X.select_dtypes("object").columns.tolist()
num_cols = X.select_dtypes("number").columns.tolist()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])
clf = make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
clf.fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]
sex_te = X_te["sex"].values
yp = np.where(sex_te == "female", proba >= 0.6, proba >= 0.5).astype(int)
res = pd.DataFrame({"sex": sex_te, "y": y_te.values, "pred": yp})
f = res[res["sex"] == "female"]
tpr_f = ((f["pred"] == 1) & (f["y"] == 1)).sum() / (f["y"] == 1).sum()
assert abs(tpr_f - 0.75) < 0.05, tpr_f
print(f"female TPR at t=0.6: {tpr_f:.4f}")


### What to learn next
- Other fairness metrics: equalized odds, demographic parity, calibration by group.
- Pre-processing mitigations: reweighting or resampling the training data to remove bias at the source.
- The full fairness toolkit: `fairlearn` and `aif360` for production-grade audits.

*Files in this folder: german_credit_prepared.csv (1000 rows: default + 21 features including sex, age, credit_amount, duration_in_month).*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
